In [1]:
# Importando as bibliotecas

import pandas as pd 
import numpy as np
import os
from datetime import datetime, date
from google.cloud import bigquery
from dotenv import load_dotenv
import pyarrow


# Carregando variáveis de ambiente
load_dotenv()

True

In [2]:
df_2020_01 = pd.read_csv("/home/danielpedro/Engenharia de Dados/combustivel_preco2015_2025/dataset/ca-2020-01.csv", sep=";")

In [3]:
df_2020_01.head()

,Regiao - Sigla,Estado - Sigla,Municipio,Revenda,CNPJ da Revenda,Nome da Rua,Numero Rua,Complemento,Bairro,Cep,Produto,Data da Coleta,Valor de Venda,Valor de Compra,Unidade de Medida,Bandeira
0,SE,SP,GUARULHOS,AUTO POSTO SAKAMOTO LTDA,49.051.667/0001-02,RODOVIA PRESIDENTE DUTRA,S/N,"KM 210,5-SENT SP/RJ",BONSUCESSO,07178-580,GASOLINA,03/01/2020,"4,399",NaN,R$ / litro,PETROBRAS DISTRIBUIDORA S.A.
1,SE,SP,GUARULHOS,AUTO POSTO SAKAMOTO LTDA,49.051.667/0001-02,RODOVIA PRESIDENTE DUTRA,S/N,"KM 210,5-SENT SP/RJ",BONSUCESSO,07178-580,ETANOL,03/01/2020,"3,199",NaN,R$ / litro,PETROBRAS DISTRIBUIDORA S.A.
2,SE,SP,GUARULHOS,AUTO POSTO SAKAMOTO LTDA,49.051.667/0001-02,RODOVIA PRESIDENTE DUTRA,S/N,"KM 210,5-SENT SP/RJ",BONSUCESSO,07178-580,DIESEL S10,03/01/2020,"3,899",NaN,R$ / litro,PETROBRAS DISTRIBUIDORA S.A.
3,SE,SP,GUARULHOS,AUTO POSTO SAKAMOTO LTDA,49.051.667/0001-02,RODOVIA PRESIDENTE DUTRA,S/N,"KM 210,5-SENT SP/RJ",BONSUCESSO,07178-580,GNV,03/01/2020,"2,995",NaN,R$ / m³,PETROBRAS DISTRIBUIDORA S.A.
4,NE,BA,SALVADOR,PETROBRAS DISTRIBUIDORA S.A.,34.274.233/0015-08,RUA EDISTIO PONDE,474,NaN,STIEP,41770-395,GASOLINA,03/01/2020,"4,69","4,1743",R$ / litro,PETROBRAS DISTRIBUIDORA S.A.


In [4]:
df_2020_02 = pd.read_csv("/home/danielpedro/Engenharia de Dados/combustivel_preco2015_2025/dataset/ca-2020-02.csv", sep=";",low_memory=False)

In [5]:
df_2020 = pd.concat((df_2020_01, df_2020_02))

In [6]:
df_2020.info()

<class 'pandas.core.frame.DataFrame'>
Index: 719300 entries, 0 to 222636
Data columns (total 16 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   Regiao - Sigla     719300 non-null  object
 1   Estado - Sigla     719300 non-null  object
 2   Municipio          719300 non-null  object
 3   Revenda            719300 non-null  object
 4   CNPJ da Revenda    719300 non-null  object
 5   Nome da Rua        719300 non-null  object
 6   Numero Rua         718781 non-null  object
 7   Complemento        184202 non-null  object
 8   Bairro             716694 non-null  object
 9   Cep                719300 non-null  object
 10  Produto            719300 non-null  object
 11  Data da Coleta     719300 non-null  object
 12  Valor de Venda     719300 non-null  object
 13  Valor de Compra    156012 non-null  object
 14  Unidade de Medida  719300 non-null  object
 15  Bandeira           719300 non-null  object
dtypes: object(16)
memory usag

In [7]:
df_2020_pe = df_2020[df_2020["Estado - Sigla"] == "PE"]

In [8]:
df_2020_pe.head()

,Regiao - Sigla,Estado - Sigla,Municipio,Revenda,CNPJ da Revenda,Nome da Rua,Numero Rua,Complemento,Bairro,Cep,Produto,Data da Coleta,Valor de Venda,Valor de Compra,Unidade de Medida,Bandeira
1681,NE,PE,ARARIPINA,FELIX COMBUSTÍVEIS ARARIPINA LTDA.,09.266.633/0001-10,AVENIDA ANTONIO DE BARROS MUNIZ,S/N,NaN,CENTRO,56280-000,GASOLINA,03/01/2020,"5,04",NaN,R$ / litro,PETROBRAS DISTRIBUIDORA S.A.
1682,NE,PE,ARARIPINA,FELIX COMBUSTÍVEIS ARARIPINA LTDA.,09.266.633/0001-10,AVENIDA ANTONIO DE BARROS MUNIZ,S/N,NaN,CENTRO,56280-000,ETANOL,03/01/2020,"3,55",NaN,R$ / litro,PETROBRAS DISTRIBUIDORA S.A.
1683,NE,PE,ARARIPINA,FELIX COMBUSTÍVEIS ARARIPINA LTDA.,09.266.633/0001-10,AVENIDA ANTONIO DE BARROS MUNIZ,S/N,NaN,CENTRO,56280-000,DIESEL S10,03/01/2020,"3,7",NaN,R$ / litro,PETROBRAS DISTRIBUIDORA S.A.
1684,NE,PE,ARARIPINA,JUVENAL ANGELO & CIA LTDA,04.965.564/0003-81,AVENIDA GOVERNADOR MUNIZ FALCAO,525,A,CENTRO,56280-000,GASOLINA,03/01/2020,"5,045",NaN,R$ / litro,RAIZEN
1685,NE,PE,ARARIPINA,JUVENAL ANGELO & CIA LTDA,04.965.564/0003-81,AVENIDA GOVERNADOR MUNIZ FALCAO,525,A,CENTRO,56280-000,ETANOL,03/01/2020,"3,65",NaN,R$ / litro,RAIZEN


In [ ]:
# Configurando credenciais

# Caminho do arquivo de credenciais GCP
credencial = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")

# Identificador do projeto no GCP
project_id = os.getenv("PROJECT_ID")

# Tabela Bronze no BigQuery
table_id = os.getenv("BRONZE_2020")

# Inicialização do cliente BigQuery

# Define a variavel de ambiente
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = credencial

# Cria o cliente Bigquery ao projeto configurado
client = bigquery.Client(project=project_id)

In [10]:
# Criação e carga da tabela de BRONZE

# Configuração do job de carga no BigQuery
# WRITE_TRUNCATE garante que a tabela Bronze seja sempre sobrescrita,
# mantendo apenas o retrato mais recente dos dados de origem
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE"
)

# Envia o DataFrame para o BigQuery,
# criando a tabela automaticamente caso ainda não exista
job = client.load_table_from_dataframe(
    df_2020_pe,
    table_id,
    job_config=job_config
)

# Aguarda a finalização do job para garantir consistência da carga
job.result()

# Confirmação visual de sucesso da operação
print("✅ Tabela Bronze criada e dados carregados com sucesso")

/home/danielpedro/anaconda3/envs/combustivel_projeto/lib/python3.11/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


✅ Tabela Bronze criada e dados carregados com sucesso
